In [82]:
import os 
import orjson
import polars as pl
import simple_icd_10_cm as cm
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
from functools import partial
from tqdm import tqdm

DATA_PATH = Path('/data/gusev/USERS/jpconnor/data/')
OncDRS_PATH = Path('/data/gusev/PROFILE/CLINICAL/OncDRS/')
PROFILE_DATA_PATH = DATA_PATH / 'PROFILE_DATA/'
PROFILE_NOTES_PATH = PROFILE_DATA_PATH / 'CLINICAL_NOTES/'
os.makedirs(PROFILE_NOTES_PATH, exist_ok=True)

def read_json_text_file(filename, schema):
    with open(filename, "rb") as f:
        data = orjson.loads(f.read())
    
    df = (pl.from_dicts(data["response"]["docs"], 
                        schema=schema,)
          .with_columns(pl.col('EVENT_DATE')
                        .str.to_datetime(format="%Y-%m-%dT%H:%M:%SZ", 
                                         time_zone='UTC')))
    return df

def file_size(path: str | Path) -> int:
    return Path(path).stat().st_size

def human_size(size: int) -> str:
    size = float(size)

    for unit in ("B", "KB", "MB", "GB", "TB"):
        if size < 1024 or unit == "TB":
            return f"{size:.1f} {unit}"
        size /= 1024

def clean_path_image_notes(text_df):
    cleaned_df = (
        text_df
        .with_columns(
            pl.col(['RPT_TEXT', 'NARRATIVE_TEXT'])
            .replace('', None)
        )
        .with_columns(
            pl.when(pl.col("RPT_TEXT").is_null())
            .then(pl.col("NARRATIVE_TEXT"))
            .when(pl.col("NARRATIVE_TEXT").is_null())
            .then(pl.col("RPT_TEXT"))
            .when(pl.col("RPT_TEXT").str.contains(pl.col("NARRATIVE_TEXT"), literal=True))
            .then(pl.col("RPT_TEXT"))
            .when(pl.col("NARRATIVE_TEXT").str.contains(pl.col("RPT_TEXT"), literal=True))
            .then(pl.col("NARRATIVE_TEXT"))
            .otherwise(
                pl.concat_str(
                    ["RPT_TEXT", "NARRATIVE_TEXT"],
                    separator="\n\n"
                )
            )
            .alias("COMBINED_TEXT")
        )
        .drop(["RPT_TEXT", "NARRATIVE_TEXT"])
        .rename({"COMBINED_TEXT": "RPT_TEXT"})
    )
    return cleaned_df
        
text_pulls = {
    'CLINICAL_TEXTS_2024_03' : {'sub_dirs' : None}, 
    'CLINICAL_TEXTS_2025_03' : {'sub_dirs' : None},
    'CLINICAL_TEXTS_2025_11' : {'sub_dirs' : None}, 
    'CLINICAL_TEXTS_2026_03' : {
        'sub_dirs' : ['Discharge Summary Notes', 'Pathology_Cytology Notes', 
                      'Progress Notes', 'Imaging Notes']}
}

image_path_schema = {'RPT_ID' : pl.Int64,
                     'DFCI_MRN' : pl.Int64, 
                     'EVENT_DATE' : pl.String, 
                     'PROC_DESC' : pl.String, 
                     'RPT_TYPE' : pl.String,
                     'RPT_TEXT' : pl.String,
                     'NARRATIVE_TEXT' : pl.String}

prog_disc_schema = {'RPT_ID' : pl.Int64,
                    'DFCI_MRN' : pl.Int64, 
                    'EVENT_DATE' : pl.String,
                    'INP_RPT_TYPE' : pl.String,
                    'PROVIDER_TYPE' : pl.String, 
                    'ENCOUNTER_TYPE_DESC' : pl.String,
                    'RPT_TEXT' : pl.String}

path_meta = pl.read_parquet(PROFILE_NOTES_PATH / 'PATHOLOGY_NOTES_METADATA.parquet')
image_meta = pl.read_parquet(PROFILE_NOTES_PATH / 'IMAGING_NOTES_METADATA.parquet')
prog_meta = pl.read_parquet(PROFILE_NOTES_PATH / 'PROGRESS_NOTES_METADATA.parquet')

path_file_counts = path_meta['FILE'].value_counts().sort('count', descending=True)
path_files_to_extract = path_file_counts['FILE'].to_list()

path_join_cols = ['RPT_ID', 'DFCI_MRN', 'EVENT_DATE', 'PROC_DESC', 'RPT_TYPE']

path_df_list = []
for path_file in tqdm(path_files_to_extract):
    path_file_meta = path_meta.filter(pl.col('FILE') == path_file)
    text_file = (
        clean_path_image_notes(
            read_json_text_file(OncDRS_PATH / path_file, image_path_schema)
        )
        .join(path_file_meta, 
              on=path_join_cols, 
              how='inner', 
              nulls_equal=True)
    )
    assert(len(text_file) == path_file_counts.filter(pl.col('FILE') == path_file)['count'][0])
    path_df_list.append(text_file)
    
complete_path_df = pl.concat(path_df_list, how='vertical')
assert(len(complete_path_df) == path_file_counts['count'].sum())
complete_path_df.write_parquet(PROFILE_NOTES_PATH / 'PATHOLOGY_NOTES.parquet', 
                               compression='zstd', compression_level=15)

source_file_size = sum([file_size(OncDRS_PATH / path_file) for path_file in path_files_to_extract])
end_file_size = file_size(PROFILE_NOTES_PATH / 'PATHOLOGY_NOTES.parquet')

print(f'source data size = {human_size(source_file_size)}')
print(f'end file size = {human_size(end_file_size)}')

del complete_path_df, path_df_list, text_file

image_file_counts = image_meta['FILE'].value_counts().sort('count', descending=True)
image_files_to_extract = image_file_counts['FILE'].to_list()

image_join_cols = ['RPT_ID', 'DFCI_MRN', 'EVENT_DATE', 'PROC_DESC', 'RPT_TYPE']

image_df_list = []
for image_file in tqdm(image_files_to_extract):
    image_file_meta = image_meta.filter(pl.col('FILE') == image_file)
    text_file = (
        clean_path_image_notes(
            read_json_text_file(OncDRS_PATH / image_file, image_path_schema)
        )
        .join(image_file_meta, 
              on=image_join_cols, 
              how='inner', 
              nulls_equal=True)
    )
    assert(len(text_file) == image_file_counts.filter(pl.col('FILE') == image_file)['count'][0])
    image_df_list.append(text_file)
    
complete_image_df = pl.concat(image_df_list, how='vertical')
assert(len(complete_image_df) == image_file_counts['count'].sum())
complete_image_df.write_parquet(PROFILE_NOTES_PATH / 'IMAGING_NOTES.parquet',
                               compression='zstd', compression_level=15)

source_file_size = sum([file_size(OncDRS_PATH / image_file) for image_file in image_files_to_extract])
end_file_size = file_size(PROFILE_NOTES_PATH / 'IMAGING_NOTES.parquet')

print(f'source data size = {human_size(source_file_size)}')
print(f'end file size = {human_size(end_file_size)}')

del complete_image_df, image_df_list, text_file

prog_file_counts = prog_meta['FILE'].value_counts().sort('count', descending=True)
prog_files_to_extract = prog_file_counts['FILE'].to_list()

prog_join_cols = ['RPT_ID', 'DFCI_MRN', 'EVENT_DATE', 'INP_RPT_TYPE', 'PROVIDER_TYPE', 'ENCOUNTER_TYPE_DESC']

prog_df_list = []
for prog_file in tqdm(prog_files_to_extract):
    prog_file_meta = prog_meta.filter(pl.col('FILE') == prog_file)
    text_file = (read_json_text_file(OncDRS_PATH / prog_file, prog_disc_schema)
                 .join(prog_file_meta, 
                       on=prog_join_cols, 
                       how='inner', 
                       nulls_equal=True)
                )
    assert(len(text_file) == prog_file_counts.filter(pl.col('FILE') == prog_file)['count'][0])
    prog_df_list.append(text_file)
    
complete_prog_df = pl.concat(prog_df_list, how='vertical')
assert(len(complete_prog_df) == prog_file_counts['count'].sum())
complete_prog_df.write_parquet(PROFILE_NOTES_PATH / 'PROGRESS_NOTES.parquet',
                              compression='zstd', compression_level=15)

source_file_size = sum([file_size(OncDRS_PATH / prog_file) for prog_file in prog_files_to_extract])
end_file_size = file_size(PROFILE_NOTES_PATH / 'PROGRESS_NOTES.parquet')

print(f'source data size = {human_size(source_file_size)}')
print(f'end file size = {human_size(end_file_size)}')

del complete_prog_df, prog_df_list, text_file